# SEARCH IMPROVE

In [5]:
import os
import sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.append(project_root)
from dotenv import load_dotenv
load_dotenv()

True

# OpenSearch test

In [13]:
from app.core.aws_clients import get_opensearch_client
from app.services.embeddings.bedrock_service import embed_text

In [8]:
os_client = get_opensearch_client()
INDEX_NAME = os.getenv('OPENSEARCH_NEW_INDEX')

In [11]:
# conteo de índices
res = os_client.count(index=INDEX_NAME)
print(res)

{'count': 466, '_shards': {'total': 5, 'successful': 5, 'skipped': 0, 'failed': 0}}


## Test de búsquedas

1. Ejemplo de match con descripciones

Para búsqueda textual. Usa análisis lingüístico (stemming, tokenización, stopwords).

In [ ]:

query = {
    "query": {
        "match": {
            "unified_description": "piscina"
        }
    }
}

res = os_client.search(index=INDEX_NAME, body=query)

for hit in res["hits"]["hits"]:
    print(f"Score: {hit['_score']}, Title: {hit['_source']['title']}, Descripción: {hit['_source']['unified_description']}")


Score: 5.0251484, Title: Casa Test 2, Descripción: Casa con patio y piscina Dormitorio Principal Habitación para descanso Baño Principal Servicios higiénicos Cocina Área de cocina
Score: 4.0042906, Title: Mini-departamento moderno en Residencial La Salle II, Breña, Descripción: Área confortable con acceso a instalaciones compartidas como piscina y gimnasio. Ideal para quienes buscan el lujo de la ciudad sin dejar de lado la tranquilidad. Dormitorio Principal Habitación para descanso Baño Principal Servicios higiénicos Sala Área social Cocina Área de cocina


2. Ejemplo de búsqueda con embeddings

In [16]:
text_example= 'Busco una casa con 3 habitaciones en los olivos, 2 baños , sala y comedor. En santiago de Surco'
text_embedding = embed_text(text_example)

In [17]:
query = {
  "size": 3,
  "query": {
    "knn": {
      "description_embedding": {
        "vector": text_embedding,
        "k": 3
      }
    }
  }
}

In [18]:
res = os_client.search(index=INDEX_NAME, body=query)

In [28]:
for hit in res["hits"]["hits"]:
    print(f"Score: {hit['_score']}, Title: {hit['_source']['title']}, Descripción: {hit['_source']['unified_description']}",
          f"Price: {hit['_source']['price']}, Geolocation: {hit['_source']['geolocation']}")

Score: 0.006636993, Title: Casa elegante en Urbanización Sol de Oro, Los Olivos, Descripción: Alquiler de una hermosa y moderna casa en Avenida Trébol, Urbanización Sol de Oro, Los Olivos. Este espacio ofrece cómodo ambiente y acceso a servicios en Lima Metropolitana. Baño Principal Servicios higiénicos Sala Área social Price: 2183.0, Geolocation: {'lat': -12.00115716048272, 'lon': -77.0670474749214}
Score: 0.0059472937, Title: Casa confortable en Urbanización Cooviecma, Santiago de Surco, Descripción: Alquila una moderna y acogedora casa en la prestigiosa urbanización Cooviecma. Ideal para vivir a menos de 10 minutos del centro de Santiago de Surco. Dormitorio Principal Habitación para descanso Dormitorio 2 Habitación para descanso Dormitorio 3 Habitación para descanso Baño Principal Servicios higiénicos Cocina Área de cocina Price: 2367.0, Geolocation: {'lat': -12.141296137272713, 'lon': -77.00641869408621}
Score: 0.0059274407, Title: Mini-departamento confortable en Urbanización El 

3. Ejemplo de búsqueda con keywords

Para exact match

In [ ]:
query = {
  "size": 5,
  "query": {
    "term": {
      "property_type": "mini-departamento"
    }
  }
}

In [40]:
res = os_client.search(index=INDEX_NAME, body=query)

In [41]:
for hit in res["hits"]["hits"]:
    print(f"Score: {hit['_score']}, Title: {hit['_source']['title']}, Descripción: {hit['_source']['unified_description']}",
          f"Price: {hit['_source']['price']}, Geolocation: {hit['_source']['geolocation']}")

Score: 1.8184278, Title: Mini-departamento lujoso en Emporio Comercial de Gamarra, Descripción: Venta de un moderno y elegante mini-departamento en el pasaje Hernando de Luque, Urbanización El Porvenir, La Victoria. Ubicación privilegiada, a pocos metros del centro comercial Emporio Comercial de Gamarra. Dormitorio Principal Habitación para descanso Baño Principal Servicios higiénicos Sala Área social Cocina Área de cocina Price: 165593.0, Geolocation: {'lat': -12.064716110801635, 'lon': -77.01643300334294}
Score: 1.8184278, Title: Moderno y confortable departamento en Lima, Descripción: Aprovecha este excelente departamento moderno, situado en un barrio tranquilo de Lima. Se ofrece una vista hermosa desde el balcón, y todo el equipo es reciente y de calidad. Dormitorio Principal Habitación para descanso Baño Principal Servicios higiénicos Sala Área social Cocina Área de cocina Price: 837.0, Geolocation: {'lat': -12.089322061121354, 'lon': -77.03509424461032}
Score: 1.8184278, Title: M

In [32]:
len(res['hits']['hits'])

10

4. Bool Queries

Usas must, should, must_not y filter para combinar condiciones:

- must: condiciones obligatorias (como AND).
- should: condiciones opcionales (como OR).
- filter: no afecta el score, pero filtra resultados.
- must_not: condiciones que deben no cumplirse (exclusión).

In [60]:
text_example= 'Departamento con balcón , elegante, moderno en un barrio tranquilo, histórico, zona céntrica'
embed_text = embed_text(text_example)

In [83]:
query = {
    "size": 5,
    "query":{
        "bool":{
            "must": [
                #{"term": { "property_type": "mini-departamento"} },
                {"term": {"operation_type": "alquiler"}},
                {"geo_distance": {"distance": "5km", "geolocation": {"lat": -11.961409039974946, "lon":  -77.07631895655102}}}
            ],
            "filter": [
                { "range": { "price": { "lte": 2000 } } }
            ],
            "should": [
                {
                    "knn": {
                        "description_embedding": {
                            "vector": embed_text,
                            "k": 20
                         }
                    }
                }
            ]
        }
    }
}

res = os_client.search(index=INDEX_NAME, body=query)

In [84]:
for hit in res["hits"]["hits"]:
    print(f"Score: {hit['_score']}, Title: {hit['_source']['title']}, Descripción: {hit['_source']['unified_description']}",
          f"Price: {hit['_source']['price']}, Geolocation: {hit['_source']['geolocation']}")

Score: 1.7565168, Title: Departamento lujoso en Los Jazmines del Naranjal, Lima, Descripción: Departamento moderno con estilo y comodidad en el exclusivo barrio Los Olivos, Lima. Amenidades incluyen jardines comunales, seguridad 24/7 y acceso a transporte público. Dormitorio Principal Habitación para descanso Baño Principal Servicios higiénicos Sala Área social Cocina Área de cocina Price: 1884.0, Geolocation: {'lat': -11.97693779607747, 'lon': -77.0781167466375}
Score: 1.7224176, Title: Mini-departamento moderno en Los Olivos, Santa Cruz, Lima, Descripción: Alquila un espacioso y cómodo mini-departamento ubicado en el corazón del distrito de Los Olivos, Santa Cruz, Lima. El departamento cuenta con una cocina moderna, sala de estar, dormitorio y baño completos. Dormitorio Principal Habitación para descanso Baño Principal Servicios higiénicos Sala Área social Cocina Área de cocina Price: 1104.0, Geolocation: {'lat': -12.004506200742703, 'lon': -77.07196294049412}
Score: 1.7224176, Title